[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/projects/02_classification/project.ipynb)

# Project 02 — Dự đoán khách hàng rời bỏ (Classification)

> Một nhà mạng muốn chạy **chiến dịch gọi điện giữ chân** khách hàng. Mỗi cuộc gọi tốn **50.000đ**; nếu gọi đúng người sắp rời bỏ, **30%** trong số họ ở lại, mỗi khách giữ được đáng giá **1.500.000đ**. Tổng đài chỉ gọi được **20%** số khách mỗi tháng. Hiện nay đội marketing gọi theo luật: "hợp đồng theo tháng **và** dùng dưới 12 tháng".

**Bạn sẽ làm**:
1. EDA hướng câu hỏi: tỉ lệ rời bỏ theo hợp đồng, thời gian sử dụng, số cuộc gọi hỗ trợ.
2. Pipeline + 3 họ mô hình (logistic, random forest, gradient boosting); so sánh bằng CV với ROC-AUC, PR-AUC, Brier.
3. Đánh giá test: AUC, calibration; **lift** của 10% khách rủi ro nhất.
4. Quyết định theo **kinh tế chiến dịch**: ngưỡng tối ưu khi không giới hạn; chọn top 20% khi có giới hạn năng lực — so với luật hiện tại.
5. Diễn giải: yếu tố nào làm tăng rủi ro; báo cáo.

**Tiêu chí đạt**: ROC-AUC test > 0.79 · lift top 10% > 2 · khi gọi được 20% khách, lợi nhuận theo mô hình cao hơn luật hiện tại ít nhất 20% · chính sách theo ngưỡng kinh tế lãi hơn "gọi tất cả".

**Chương liên quan**: 07 (logistic), 11–12 (rừng, boosting), 14 (metric, ROC/PR, ngưỡng), 15 (CV), 19 (mất cân bằng, chi phí), 36 (giải thích), 37 (calibration).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/projects/02_classification

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.utils import repo_path

customers = pd.read_csv(repo_path("data", "sample", "customer_churn.csv"))
CALL_COST, CUSTOMER_VALUE, SUCCESS_RATE, CAPACITY = 50_000, 1_500_000, 0.30, 0.20
print(customers.shape, f"tỉ lệ rời bỏ: {customers.churn.mean():.1%}")
customers.head()

## Bước 1 — EDA hướng câu hỏi

Tính tỉ lệ rời bỏ theo `contract` → `churn_by_contract` (Series), theo nhóm `tenure_months` (`[0, 6)`, `[6, 12)`, `[12, 24)`, `[24, 48)`, `[48, 73)`) → `churn_by_tenure`, và theo `support_calls` (gộp ≥ 5 thành 5) → `churn_by_calls`. Vẽ ba biểu đồ cột.

In [ ]:
### START CODE HERE ### (≈ 6 dòng)
churn_by_contract = None
churn_by_tenure = None
churn_by_calls = None
fig, axes = None, None
### END CODE HERE ###
plt.tight_layout()
plt.show()

In [ ]:
check_greater(churn_by_contract["month-to-month"], 3 * churn_by_contract["two_year"], "hợp đồng tháng rời bỏ nhiều hơn hẳn")
check_greater(churn_by_tenure.iloc[0], churn_by_tenure.iloc[-1], "khách mới rời bỏ nhiều hơn khách lâu năm")
check_true(churn_by_calls.is_monotonic_increasing, "càng gọi hỗ trợ nhiều càng dễ rời bỏ")

## Bước 2 — Chia dữ liệu và so sánh mô hình

Chia 75/25 có phân tầng (`random_state=42`). Tạo pipeline chung (chuẩn hoá biến số, one-hot biến phân loại) và 3 mô hình:
`logreg` (`LogisticRegression(max_iter=1000)`), `forest` (`RandomForestClassifier(300, min_samples_leaf=10, random_state=0, n_jobs=-1)`), `boosting` (`HistGradientBoostingClassifier(learning_rate=0.05, max_leaf_nodes=15, random_state=0)`). So sánh bằng 5-fold CV phân tầng → `cv_table` (DataFrame index = tên mô hình, cột `roc_auc`, `pr_auc`, `brier`).

In [ ]:
NUMERIC = ["tenure_months", "monthly_charges", "total_charges", "support_calls", "n_products"]
CATEGORICAL = ["contract", "internet_service", "payment_method", "senior"]
X, y = customers[NUMERIC + CATEGORICAL], customers.churn
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)


def preprocess():
    return ColumnTransformer([("num", StandardScaler(), NUMERIC), ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])


### START CODE HERE ### (≈ 10 dòng)
models, cv_table = {}, None
### END CODE HERE ###
print(cv_table.round(4))

In [ ]:
check_equal(sorted(cv_table.index), ["boosting", "forest", "logreg"], "ba mô hình")
check_equal(list(cv_table.columns), ["roc_auc", "pr_auc", "brier"], "ba metric")
check_greater(cv_table.roc_auc.min(), 0.78, "mọi mô hình đều học được")
check_less(cv_table.loc["logreg", "brier"], cv_table.loc["boosting", "brier"] + 0.002, "logistic hiệu chỉnh tốt ít nhất ngang boosting")

Ba mô hình gần nhau; logistic nhỉnh hơn ở cả ba metric — dữ liệu này có quan hệ gần tuyến tính trong log-odds. Chọn **logistic**: tốt nhất, hiệu chỉnh tốt (quan trọng vì quyết định dùng xác suất), dễ giải thích cho đội marketing.

## Bước 3 — Đánh giá test

Huấn luyện `logreg` trên train → xác suất test `p_test`; tính `auc_test`, `brier_test`; **lift** của 10% khách có xác suất cao nhất `lift_top10` (= tỉ lệ rời bỏ trong top 10% ÷ tỉ lệ rời bỏ chung). Lập bảng calibration theo thập phân vị `calibration` (cột `predicted`, `actual`).

In [ ]:
### START CODE HERE ### (≈ 7 dòng)
final = None
p_test = None
auc_test = None
brier_test = None
top10 = None
lift_top10 = None
calibration = None
### END CODE HERE ###
print(f"ROC-AUC = {auc_test:.3f} | PR-AUC = {average_precision_score(y_te, p_test):.3f} | Brier = {brier_test:.4f} | lift top 10% = {lift_top10:.2f}")
print(calibration.round(3).T.to_string())

In [ ]:
check_greater(auc_test, 0.79, "ROC-AUC test")
check_greater(lift_top10, 2.0, "lift top 10%")
check_less((calibration.predicted - calibration.actual).abs().max(), 0.1, "xác suất hiệu chỉnh tốt ở mọi thập phân vị")

## Bước 4 — Quyết định theo kinh tế chiến dịch

Gọi khách có xác suất rời bỏ $p$: lợi kỳ vọng $= p \cdot 0.30 \cdot 1.5\text{tr} - 50\text{k}$ → gọi khi $p > \dfrac{50\text{k}}{0.3 \cdot 1.5\text{tr}} \approx 11\%$.

Viết `campaign_profit(call, churned)` (đồng): mỗi người được gọi trừ chi phí 50k; mỗi người được gọi **và** rời bỏ thật mang lại $0.3 \times 1.5$tr (kỳ vọng). Tính lợi nhuận trên test của:
- `profit_threshold`: gọi khi $p >$ ngưỡng kinh tế (không giới hạn năng lực) · `profit_all`: gọi tất cả;
- `profit_model_top20`: gọi 20% khách có $p$ cao nhất · `profit_rule`: luật hiện tại (hợp đồng tháng **và** `tenure_months < 12`).

In [ ]:
### START CODE HERE ### (≈ 8 dòng)
def campaign_profit(call, churned):
    return 0.0
threshold = None
profit_threshold, profit_all, profit_model_top20, profit_rule, rule = None, None, None, None, None
### END CODE HERE ###
print(f"không giới hạn — gọi theo ngưỡng {threshold:.1%}: {profit_threshold / 1e6:.1f} tr (gọi {np.mean(p_test > threshold):.0%}) | gọi tất cả: {profit_all / 1e6:.1f} tr")
print(f"giới hạn 20%   — mô hình top 20%: {profit_model_top20 / 1e6:.1f} tr | luật hiện tại (gọi {rule.mean():.0%}): {profit_rule / 1e6:.1f} tr")

In [ ]:
check_close(campaign_profit([True, True, False], [1, 0, 1]), 0.3 * 1_500_000 - 2 * 50_000, message="công thức lợi nhuận")
check_greater(profit_threshold, profit_all, "ngưỡng kinh tế lãi hơn gọi tất cả")
check_greater(profit_model_top20, 1.2 * profit_rule, "cùng năng lực, mô hình lãi hơn luật ≥ 20%")

## Bước 5 — Diễn giải

Dùng permutation importance (ROC-AUC, 10 lần lặp, `random_state=0`) trên **test** với **dữ liệu gốc** (trước one-hot) → Series `importance` (sắp giảm dần).

In [ ]:
### START CODE HERE ### (≈ 2 dòng)
result = None
importance = None
### END CODE HERE ###
print(importance.round(4).to_string())

In [ ]:
check_equal(len(importance), 9, "9 đặc trưng gốc")
check_true(importance.index[0] in {"contract", "tenure_months", "support_calls"}, "yếu tố quan trọng nhất hợp lý")
check_less(importance["senior"], importance["contract"], "người cao tuổi ít quan trọng hơn hợp đồng")

## Bước 6 — Báo cáo

Viết 5–8 câu cho trưởng phòng marketing: mô hình nào, tốt hơn luật hiện tại bao nhiêu **tiền**, nên gọi ai, rủi ro/giả định (uplift chưa đo), bước tiếp theo (A/B test, giám sát).

✍️ Báo cáo: